# EX 2:  Continual pretraining: Comparaison des tailles de vocabulaire étendu (LoRA)

**Modèle :** `SmolLM3-3B-Base` · **Données :** wolof monolingue · **Méthode :** QLoRA 4-bit

Ce notebook exécute **deux expériences** dans la même session, avec des paramètres
strictement identiques à l'exception de la taille du vocabulaire :

| Expérience | Tokenizer | Vocabulaire |
|---|---|---|
| `wolof-ext8000-lora` | étendu 8 000 | 136 256 |
| `wolof-ext4000-lora` | étendu 4 000 | 132 256 |

## Question posée

Les mesures **avant entraînement** montraient une divergence entre les deux métriques :

| Tokens ajoutés | Fertilité | BPB initial |
|---|---|---|
| 4 000 | 1,295 | **2,1797** |
| 8 000 | **1,265** | 2,2478 |

Le vocabulaire 8 000 compresse mieux mais modélise moins bien. **Cet écart survit-il à l'entraînement ?**
Les embeddings mal initialisés du 8 000 peuvent converger, et le classement s'inverser.

## Points de méthode

- **Le modèle initialisé est chargé depuis le disque**, jamais reconstruit. La chirurgie des
  embeddings (déplacement des tokens spéciaux, *mean-pooling*, remapping des identifiants)
  a été faite une fois dans le notebook `tokenizer_extension`. La refaire ici risquerait
  de produire deux variantes légèrement différentes, et surtout d'oublier le déplacement
  des lignes spéciales, qui laisserait `eos` avec un embedding aléatoire.
- **`modules_to_save=["embed_tokens","lm_head"]`** est indispensable : sans lui, LoRA gèle
  les embeddings et les nouveaux tokens wolof ne seraient **jamais** entraînés — l'expérience
  échouerait en silence.
- **Un seul learning rate** pour les deux runs. La comparaison porte sur le vocabulaire ;
  tout autre paramètre doit rester constant.
- **L'évaluation utilise le split propre** (`wolof_eval_clean_v1.parquet`), débarrassé des
  quasi-doublons présents dans le train.

## 1 Environnement

In [ ]:
!pip install -q -U bitsandbytes accelerate datasets pandas matplotlib peft transformers

## 2  Configuration commune

Tous les hyperparamètres ci-dessous sont **partagés par les deux expériences**.

In [ ]:
ROOT      = "/content/drive/MyDrive/SmolLM3-3B-Base"
BASE_ID   = "HuggingFaceTB/SmolLM3-3B-Base"
GPU       = "L4-22Go"        # à adapter


MAX_STEPS = 250              
SEQ_LEN   = 1024
MICRO_BS  = 4
GRAD_ACC  = 8                
LR        = 2e-4            
SEED      = 42

TOKENS_PAR_PAS = MICRO_BS * GRAD_ACC * SEQ_LEN
print(f"{TOKENS_PAR_PAS:,} tokens/pas | {MAX_STEPS} pas = {MAX_STEPS*TOKENS_PAR_PAS/1e6:.1f} M tokens")

In [ ]:
EXPERIENCES = [
    {"nom": "wolof-ext8000-lora",
     "tok_dir":   f"{ROOT}/tokenizers/smollm3_wolof_ext_8000",
     "model_dir": f"{ROOT}/models/smollm3-wolof-ext8000-init"},
    {"nom": "wolof-ext4000-lora",
     "tok_dir":   f"{ROOT}/tokenizers/smollm3_wolof_ext_4000",
     "model_dir": f"{ROOT}/models/smollm3-wolof-ext4000-init"},
]

import os
for e in EXPERIENCES:
    for k in ("tok_dir", "model_dir"):
        assert os.path.isdir(e[k]), f"introuvable : {e[k]}"
print("artefacts présents")

## 3  Données

Les splits ont été figés dans le notebook `tokenizer_extension` et ne sont **jamais régénérés**.


In [ ]:
import pandas as pd
from datasets import Dataset

train_df   = pd.read_parquet(f"{ROOT}/data/wolof_bpe_train_v1.parquet")
eval_df    = pd.read_parquet(f"{ROOT}/data/wolof_eval_clean_v1.parquet")   # split PROPRE

TEXTES_TRAIN = train_df["text"].astype(str).tolist()
TEXTES_EVAL  = eval_df["text"].astype(str).tolist()
print(f"train : {len(TEXTES_TRAIN):,} textes | éval propre : {len(TEXTES_EVAL):,} textes")

def paqueter(textes, tokenizer, longueur):
    ids = []
    for t in textes:
        ids.extend(tokenizer.encode(t, add_special_tokens=False) + [tokenizer.eos_token_id])
    n = (len(ids) // longueur) * longueur
    blocs = [ids[i:i+longueur] for i in range(0, n, longueur)]
    return Dataset.from_dict({"input_ids": blocs, "labels": [b[:] for b in blocs]})

## 4  Fonctions d'expérience


In [ ]:
import torch, math, time, gc, json
from torch.nn import functional as F
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, default_data_collator)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def charger(tok_dir, model_dir):
    """Charge le tokenizer étendu et le modèle DÉJÀ initialisé (mean-pooling + remapping)."""
    tok = AutoTokenizer.from_pretrained(tok_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(model_dir, quantization_config=bnb,
                                                 device_map={"": 0})
    model.config.use_cache = False

    # contrôles : le modèle sauvegardé doit déjà être cohérent avec son tokenizer
    assert model.config.vocab_size == len(tok), \
        f"incohérence vocab : modèle {model.config.vocab_size} vs tokenizer {len(tok)}"
    eos = model.config.eos_token_id
    eos = eos[0] if isinstance(eos, list) else eos
    print(f"  vocab={len(tok):,} | eos_id={eos} → {tok.decode([eos])!r}")
    return tok, model

### Configuration LoRA

`modules_to_save=["embed_tokens","lm_head"]` rend la matrice d'embeddings entraînable.

Deux conséquences à connaître :
- Les embeddings de SmolLM3 sont *tied* (une seule matrice sert d'entrée et de sortie).
  Passer les deux modules les **dissocie** : PEFT en crée deux copies indépendantes.
- Ces copies sont volumineuses et entraînées en pleine précision  c'est le poste
  de mémoire dominant de ces runs.

In [ ]:
def preparer_lora(model):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.gradient_checkpointing_enable()
    cfg = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        modules_to_save=["embed_tokens", "lm_head"],   # sinon les tokens wolof n'apprennent JAMAIS
    )
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

### Évaluation bits-per-byte

Le **BPB** normalise l'entropie croisée par les **octets du texte brut**, pas par les tokens.
C'est la seule métrique valide pour comparer des modèles aux tokenizers différents :
le vocabulaire 8 000 produit moins de tokens que le 4 000 sur le même texte, donc leurs
perplexités ne sont pas commensurables.

La perplexité est conservée à titre indicatif, comparable uniquement à tokenizer identique.

In [ ]:
@torch.no_grad()
def evaluer_bpb(model, tokenizer, textes, max_len=1024):
    model.eval()
    nats = octets = tokens = 0
    for t in textes:
        t = str(t)
        b = len(t.encode("utf-8"))
        if b == 0: continue
        ids = tokenizer.encode(t, add_special_tokens=False)[:max_len]
        if len(ids) < 2: continue
        x = torch.tensor([ids], device="cuda:0")
        logits = model(x).logits[0, :-1]
        nats   += F.cross_entropy(logits.float(), x[0, 1:], reduction="sum").item()
        octets += b
        tokens += len(ids) - 1
    return {"bpb": nats/(math.log(2)*octets), "ppl": math.exp(nats/tokens),
            "n_octets": octets, "n_tokens": tokens}

In [ ]:
def lancer(exp):
    nom = exp["nom"]
    out = f"{ROOT}/runs/{nom}"
    os.makedirs(out, exist_ok=True)
    print(f"\n{'='*70}\n  {nom}\n{'='*70}")

    tok, model = charger(exp["tok_dir"], exp["model_dir"])

    # BPB avant entraînement (référence propre à ce tokenizer)
    bpb_avant = evaluer_bpb(model, tok, TEXTES_EVAL[:500])
    print(f"  BPB avant entraînement : {bpb_avant['bpb']:.4f}")

    model = preparer_lora(model)

    ds_train = paqueter(TEXTES_TRAIN, tok, SEQ_LEN).shuffle(seed=SEED)
    ds_eval  = paqueter(TEXTES_EVAL,  tok, SEQ_LEN)
    print(f"  blocs — train {len(ds_train):,} | éval {len(ds_eval):,}")

    args = TrainingArguments(
        output_dir=out, max_steps=MAX_STEPS,
        per_device_train_batch_size=MICRO_BS, gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
        logging_steps=25, eval_strategy="steps", eval_steps=50,
        save_steps=MAX_STEPS, save_total_limit=1,
        bf16=True, fp16=False, gradient_checkpointing=True,
        report_to="none", seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds_train,
                      eval_dataset=ds_eval, data_collator=default_data_collator)

    t0 = time.time()
    res = trainer.train()
    duree = (time.time() - t0) / 3600

    bpb_apres = evaluer_bpb(model, tok, TEXTES_EVAL[:500])
    print(f"  BPB après entraînement : {bpb_apres['bpb']:.4f}  "
          f"(gain {100*(1-bpb_apres['bpb']/bpb_avant['bpb']):.1f} %)")

    hist  = trainer.state.log_history
    ltr   = [(h["step"], h["loss"])      for h in hist if "loss" in h]
    lval  = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

    journal = {
        "run": nom, "gpu": GPU, "modele_initialise": exp["model_dir"],
        "tokenizer": exp["tok_dir"], "vocab": len(tok),
        "donnees": "wolof monolingue", "methode": "QLoRA 4-bit + embeddings entrainables",
        "hyperparametres": {
            "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
            "modules_to_save": ["embed_tokens", "lm_head"],
            "lr": LR, "scheduler": "cosine", "warmup_ratio": 0.03,
            "seq_len": SEQ_LEN, "micro_bs": MICRO_BS, "grad_acc": GRAD_ACC,
            "batch_effectif": MICRO_BS*GRAD_ACC, "max_steps": MAX_STEPS, "seed": SEED},
        "budget": {"tokens_vus_M": MAX_STEPS*TOKENS_PAR_PAS/1e6,
                   "duree_heures": round(duree, 3),
                   "epochs_equivalent": round(MAX_STEPS*MICRO_BS*GRAD_ACC/len(ds_train), 4)},
        "loss": {"train_premiere": ltr[0][1], "train_derniere": ltr[-1][1],
                 "val_derniere": lval[-1][1] if lval else None,
                 "moyenne_run": res.metrics.get("train_loss"),
                 "courbe_train": ltr, "courbe_val": lval},
        "eval": {"split": "wolof_eval_clean_v1.parquet",
                 "bpb_avant": bpb_avant["bpb"], "bpb_apres": bpb_apres["bpb"],
                 "ppl_avant": bpb_avant["ppl"], "ppl_apres": bpb_apres["ppl"]},
    }
    json.dump(journal, open(f"{out}/journal.json", "w"), indent=2, ensure_ascii=False)
    model.save_pretrained(f"{out}/adapter")
    tok.save_pretrained(f"{out}/tokenizer")
    print(f"  sauvegardé : {out}")

    del model, trainer, ds_train, ds_eval
    gc.collect(); torch.cuda.empty_cache()
    libre, total = torch.cuda.mem_get_info()
    print(f"  VRAM libérée : {libre/1e9:.1f} / {total/1e9:.1f} Go")
    return journal

## 5  Expérience 1 : vocabulaire 8 000 + LoRA

In [ ]:
journal_8k = lancer(EXPERIENCES[0])

## 6  Expérience 2 : vocabulaire 4 000 + LoRA



In [ ]:
journal_4k = lancer(EXPERIENCES[1])

## 7  Comparaison

La colonne décisive est **BPB après**. La perplexité n'est pas comparable entre les deux
lignes, les tokenizers étant différents.

In [ ]:
import pandas as pd

lignes = []
for j in [journal_8k, journal_4k]:
    lignes.append({
        "run": j["run"], "vocab": j["vocab"],
        "BPB avant": round(j["eval"]["bpb_avant"], 4),
        "BPB après": round(j["eval"]["bpb_apres"], 4),
        "gain %": round(100*(1 - j["eval"]["bpb_apres"]/j["eval"]["bpb_avant"]), 1),
        "loss train": round(j["loss"]["train_derniere"], 3),
        "loss val": round(j["loss"]["val_derniere"], 3) if j["loss"]["val_derniere"] else None,
        "durée h": j["budget"]["duree_heures"],
    })

tableau = pd.DataFrame(lignes)
os.makedirs(f"{ROOT}/results", exist_ok=True)
tableau.to_csv(f"{ROOT}/results/comparaison_vocab_4k_8k.csv", index=False)
print(tableau.to_string(index=False))

meilleur = tableau.loc[tableau["BPB après"].idxmin()]
print(f"\nMeilleur BPB : {meilleur['run']} ({meilleur['BPB après']:.4f})")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for j, couleur in [(journal_8k, "#1f77b4"), (journal_4k, "#d62728")]:
    st, lo = zip(*j["loss"]["courbe_train"])
    ax[0].plot(st, lo, marker="o", ms=3, color=couleur, label=f"{j['run']} (train)")
    if j["loss"]["courbe_val"]:
        sv, lv = zip(*j["loss"]["courbe_val"])
        ax[0].plot(sv, lv, marker="s", ms=4, ls="--", color=couleur, alpha=.6,
                   label=f"{j['run']} (val)")
ax[0].set_xlabel("pas"); ax[0].set_ylabel("loss")
ax[0].set_title("Loss — attention : non comparable entre tokenizers")
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

noms = [j["run"] for j in [journal_8k, journal_4k]]
av   = [j["eval"]["bpb_avant"] for j in [journal_8k, journal_4k]]
ap   = [j["eval"]["bpb_apres"] for j in [journal_8k, journal_4k]]
x = range(len(noms))
ax[1].bar([i-0.2 for i in x], av, 0.4, label="avant", color="#bbb")
ax[1].bar([i+0.2 for i in x], ap, 0.4, label="après", color="#2ca02c")
ax[1].set_xticks(list(x)); ax[1].set_xticklabels(noms, fontsize=8)
ax[1].set_ylabel("BPB"); ax[1].set_title("BPB — métrique comparable")
ax[1].legend(); ax[1].grid(alpha=.3, axis="y")

plt.tight_layout()
plt.savefig(f"{ROOT}/results/comparaison_vocab_4k_8k.png", dpi=200, bbox_inches="tight")
plt.show()

## Lecture des résultats

| Issue | Interprétation |
|---|---|
| Le 4 000 garde son avance | La divergence fertilité / modélisation **survit à l'entraînement** : optimiser la fertilité au-delà d'un seuil dégrade le modèle. |
| Le 8 000 rattrape ou dépasse | Le déficit initial venait d'une **initialisation imparfaite**, corrigée par l'entraînement. La meilleure fertilité devient alors le bon critère. |
| Écart négligeable (< 0,01) | La taille du vocabulaire importe peu dans cette plage ; choisir le plus petit, moins coûteux en embeddings à entraîner. |